In [4]:
import numpy as np
import random
import math
import time
import concurrent.futures
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, log_loss

# Import file Logic_game của bạn để can thiệp trực tiếp vào trọng số
import Logic_game
from Logic_game import CaroLogic

# =====================================================================
# CẤU HÌNH VÒNG LẶP TIẾN HÓA (GENERATIONAL LEARNING)
# =====================================================================
NUM_GENERATIONS = 3         # Số thế hệ tiến hóa (Chạy 3 vòng là đủ khôn)
GAMES_PER_GEN = 100         # Số ván cờ mô phỏng trong MỖI thế hệ
MAX_MOVES_PER_GAME = 150     # Ngắt sớm ở 60 nước để lấy data tinh hoa nhất
DEPTH_A = 3                 # Độ sâu AI phe 1
DEPTH_B = 3                # Độ sâu AI phe -1

ATTACK_PATTERNS = [
    (1, 1, 1, 1, 1),       # index 0: Ăn 5
    (0, 1, 1, 1, 1, 0),    # index 1: 4 thoáng
    (0, 1, 1, 1, 1),       # index 2: 4 chặn 1 đầu
    (1, 1, 1, 1, 0),       # index 3: 4 chặn 1 đầu
    (0, 1, 1, 1, 0),       # index 4: 3 thoáng
    (0, 1, 1, 0, 1, 0),    # index 5: 3 thoáng có lỗ
    (0, 1, 0, 1, 1, 0),    # index 6: 3 thoáng có lỗ
    (0, 1, 1, 1),          # index 7: 3 chặn
    (1, 1, 1, 0),          # index 8: 3 chặn
    (0, 1, 1, 0)           # index 9: 2 thoáng
]

DEFEND_PATTERNS = [
    (-1, -1, -1, -1, -1),   # index 10
    (0, -1, -1, -1, -1, 0), # index 11
    (0, -1, -1, -1, -1),    # index 12
    (-1, -1, -1, -1, 0),    # index 13
    (0, -1, -1, -1, 0),     # index 14
    (0, -1, -1, 0, -1, 0),  # index 15
    (0, -1, 0, -1, -1, 0),  # index 16
    (0, -1, -1, -1),        # index 17
    (-1, -1, -1, 0),        # index 18
    (0, -1, -1, 0)          # index 19
]

# =====================================================================
# CÁC HÀM TRÍCH XUẤT ĐẶC TRƯNG
# =====================================================================
def count_pattern_occurrences(board, pattern):
    count = 0
    size = len(board)
    p_len = len(pattern)

    for r in range(size):
        for c in range(size):
            if c <= size - p_len:
                if tuple([board[r][c+i] for i in range(p_len)]) == pattern: count += 1
            if r <= size - p_len:
                if tuple([board[r+i][c] for i in range(p_len)]) == pattern: count += 1
            if r <= size - p_len and c <= size - p_len:
                if tuple([board[r+i][c+i] for i in range(p_len)]) == pattern: count += 1
            if r >= p_len - 1 and c <= size - p_len:
                if tuple([board[r-i][c+i] for i in range(p_len)]) == pattern: count += 1
    return count

def extract_features(board, player):
    features = [0] * 20
    normalized_board = [
        [board[r][c] if player == 1 else -board[r][c] for c in range(len(board))]
        for r in range(len(board))
    ]
    for idx, pattern in enumerate(ATTACK_PATTERNS):
        features[idx] = count_pattern_occurrences(normalized_board, pattern)
    for idx, pattern in enumerate(DEFEND_PATTERNS):
        features[10 + idx] = count_pattern_occurrences(normalized_board, pattern)
    return features

# =====================================================================
# HÀM GIẢ LẬP 1 VÁN CỜ
# =====================================================================
def play_one_game(game_id):
    logic = CaroLogic(board_size=10, bot_depth=2)
    logic.reset_game()
    game_active = True

    turn = random.choice([1, -1]) # Đảo người đi trước ngẫu nhiên
    X_history, player_history = [], []
    winner = 0
    move_count = 0

    num_random_openings = random.randint(2, 4) # Khai cuộc ngẫu nhiên

    while game_active and move_count < MAX_MOVES_PER_GAME:
        move = None

        if move_count < num_random_openings:
            center = logic.board_size // 2
            while True:
                r = center + random.randint(-3, 3)
                c = center + random.randint(-3, 3)
                if 0 <= r < logic.board_size and 0 <= c < logic.board_size and logic.board[r][c] == 0:
                    move = (r, c)
                    break
        else:
            is_max = (turn == 1)
            depth_to_use = DEPTH_A if turn == 1 else DEPTH_B
            # Luôn gọi heuristic LOGISTIC (sẽ được tự động update qua từng thế hệ)
            _, move = logic.minimax(logic.board, depth=depth_to_use, alpha=-math.inf, beta=math.inf, maximizing_player=is_max, heuristic_type="LOGISTIC")

        if not move: break

        r, c = move
        success, is_win, _ = logic.play_move(r, c, turn)

        if success:
            features = extract_features(logic.board, turn)
            X_history.append(features)
            player_history.append(turn)
            move_count += 1

        if is_win:
            winner = turn
            game_active = False
        else:
            turn = -1 if turn == 1 else 1

    # Gán nhãn hồi tố
    X_local, y_local = [], []
    for i in range(len(X_history)):
        X_local.append(X_history[i])
        if winner != 0 and player_history[i] == winner:
            y_local.append(1)
        else:
            y_local.append(0)

    # In log mượt mà
    if winner == 1: result_str = "Phe 1 thắng"
    elif winner == -1: result_str = "Phe -1 thắng"
    else: result_str = "Hòa"

    print(f"   -> Đã mô phỏng ván {game_id}/{GAMES_PER_GEN} ({result_str} - {move_count} bước).")

    return X_local, y_local, winner

# =====================================================================
# HÀM CHẠY ĐA LUỒNG & HUẤN LUYỆN (CHO 1 THẾ HỆ)
# =====================================================================
def run_generation(gen_number):
    print(f"\n" + "=".center(60, "="))
    print(f" BẮT ĐẦU THẾ HỆ TIẾN HÓA THỨ {gen_number} ".center(60))
    print("=".center(60, "="))

    X_data, y_data = [], []
    win_1 = 0
    win_minus1 = 0
    draws = 0

    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = executor.map(play_one_game, range(1, GAMES_PER_GEN + 1))
        for X_local, y_local, winner in results:
            X_data.extend(X_local)
            y_data.extend(y_local)

            # Đếm kết quả
            if winner == 1: win_1 += 1
            elif winner == -1: win_minus1 += 1
            else: draws += 1

    # IN BẢNG THỐNG KÊ TỈ LỆ
    print("\n" + "-"*60)
    print(f" THỐNG KÊ KẾT QUẢ THI ĐẤU (THẾ HỆ {gen_number}) ".center(60))
    print("-"*60)
    print(f" -> Phe 1 Thắng : {win_1} ván ({win_1/GAMES_PER_GEN*100:.1f}%)")
    print(f" -> Phe -1 Thắng: {win_minus1} ván ({win_minus1/GAMES_PER_GEN*100:.1f}%)")
    print(f" -> Hòa         : {draws} ván ({draws/GAMES_PER_GEN*100:.1f}%)")
    print("-"*60)

    X = np.array(X_data)
    y = np.array(y_data)

    if len(np.unique(y)) < 2:
        print("Cảnh báo: Thế hệ này hòa quá nhiều hoặc chỉ 1 phe thắng. Bỏ qua train.")
        return None, None

    # Train model
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    model = LogisticRegression(solver='lbfgs', max_iter=3000, fit_intercept=True, class_weight='balanced', random_state=42)
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_prob = model.predict_proba(X_test)

    bce_loss = log_loss(y_test, y_pred_prob)
    f1_win = f1_score(y_test, y_pred_test)

    print(f"\n[Gen {gen_number} Metrics] Log-Loss: {bce_loss:.4f} | F1-Score: {f1_win:.4f}")

    # Xử lý trọng số
    raw_weights = model.coef_[0]
    max_val = np.max(np.abs(raw_weights[1:])) if np.max(np.abs(raw_weights[1:])) > 0 else 1.0
    scaled_weights = (np.abs(raw_weights) / max_val) * 80000

    new_attack = {ATTACK_PATTERNS[0]: 200000.0}
    for i in range(1, len(ATTACK_PATTERNS)):
        new_attack[ATTACK_PATTERNS[i]] = max(10.0, scaled_weights[i])

    new_defend = {DEFEND_PATTERNS[0]: -200000.0}
    for i in range(1, len(DEFEND_PATTERNS)):
        val = max(10.0, scaled_weights[10 + i])
        if i == 1: val *= 2.5 # Ép phạt thế 4 thoáng
        new_defend[DEFEND_PATTERNS[i]] = -val

    return new_attack, new_defend

# =====================================================================
# VÒNG LẶP CHÍNH CỦA CHƯƠNG TRÌNH
# =====================================================================
if __name__ == "__main__":
    start_time = time.time()

    # Ở Gen 1, AI sẽ dùng chính bộ điểm bằng tay (hoặc bộ điểm cũ của bạn) để đánh mồi
    # Khởi tạo ma trận điểm tổng hợp vào thư viện Logic_game để các Worker đọc được
    Logic_game.SCORE_MATRIX_LOGISTIC.clear()
    Logic_game.SCORE_MATRIX_LOGISTIC.update(Logic_game.SCORE_MATRIX_MANUAL)

    final_attack = {}
    final_defend = {}

    for gen in range(1, NUM_GENERATIONS + 1):
        atk, dfd = run_generation(gen)

        if atk is not None and dfd is not None:
            final_attack = atk
            final_defend = dfd

            # CẬP NHẬT TRỌNG SỐ VÀO LOGIC GAME ĐỂ GEN TIẾP THEO SỬ DỤNG
            Logic_game.SCORE_MATRIX_LOGISTIC.clear()
            Logic_game.SCORE_MATRIX_LOGISTIC.update(final_attack)
            Logic_game.SCORE_MATRIX_LOGISTIC.update(final_defend)
            print(f"-> Đã Bơm trọng số thông minh của Gen {gen} vào não AI để chuẩn bị cho Gen {gen+1}!")

    print("\n" + "#"*60)
    print(" QUÁ TRÌNH TIẾN HÓA HOÀN TẤT! COPY BỘ SỐ NÀY VÀO LOGIC_GAME.PY ")
    print("#"*60)

    print("\nATTACK_MATRIX_LOGISTIC = {")
    for k, v in final_attack.items():
        print(f"    {k}: {v:.2f},")
    print("}")

    print("\nDEFEND_MATRIX_LOGISTIC = {")
    for k, v in final_defend.items():
        print(f"    {k}: {v:.2f},")
    print("}")

    print(f"\nTổng thời gian tiến hóa qua {NUM_GENERATIONS} thế hệ: {time.time() - start_time:.2f} giây")


               BẮT ĐẦU THẾ HỆ TIẾN HÓA THỨ 1                
   -> Đã mô phỏng ván 2/100 (Phe 1 thắng - 17 bước).
   -> Đã mô phỏng ván 1/100 (Phe -1 thắng - 18 bước).
   -> Đã mô phỏng ván 4/100 (Phe -1 thắng - 24 bước).
   -> Đã mô phỏng ván 3/100 (Phe 1 thắng - 26 bước).
   -> Đã mô phỏng ván 6/100 (Phe 1 thắng - 36 bước).
   -> Đã mô phỏng ván 5/100 (Phe -1 thắng - 24 bước).
   -> Đã mô phỏng ván 8/100 (Phe 1 thắng - 9 bước).
   -> Đã mô phỏng ván 9/100 (Phe -1 thắng - 17 bước).
   -> Đã mô phỏng ván 7/100 (Phe 1 thắng - 33 bước).
   -> Đã mô phỏng ván 11/100 (Phe -1 thắng - 19 bước).
   -> Đã mô phỏng ván 10/100 (Phe -1 thắng - 20 bước).
   -> Đã mô phỏng ván 12/100 (Phe 1 thắng - 17 bước).
   -> Đã mô phỏng ván 13/100 (Phe 1 thắng - 10 bước).
   -> Đã mô phỏng ván 14/100 (Phe 1 thắng - 11 bước).
   -> Đã mô phỏng ván 16/100 (Phe -1 thắng - 35 bước).
   -> Đã mô phỏng ván 17/100 (Phe 1 thắng - 9 bước).
   -> Đã mô phỏng ván 18/100 (Phe -1 thắng - 19 bước).
   -> Đã mô phỏng ván 1